# Tutorial 02 — Energy Calculation & Network Control Theory

**GSoC 2026 Project #39 — NeuroSim | INCF / EBRAINS**

This notebook demonstrates:
1. Estimating a stable directed adjacency matrix (A) via Spectral Inversion
2. Computing the Discrete-Time Controllability Gramian
3. Computing minimum control energy for brain state transitions
4. Mapping pairwise transition energies between cognitive states

> **Dr. Agarwal's Challenge addressed here:**
> *MVAR can become computationally unstable on dense parcellations. This notebook
> demonstrates that both solvers produce Schur-stable A matrices with spectral radius < 1.*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from neurosim.connectivity import (
    spectral_inversion_solver,
    mvar_solver,
    check_schur_stability,
    normalize_matrix,
)
from neurosim.control.gramian import compute_gramian
from neurosim.control.energy import minimum_energy, optimal_control_path

rng = np.random.default_rng(seed=42)
print("NeuroSim modules loaded successfully.")

## Step 1: Simulate a Synthetic Connectome

In a real pipeline this comes from a DWI tractography pipeline (e.g., MRtrix3 + Schaefer atlas).
We simulate a symmetric FC matrix representing resting-state functional connectivity.

In [ ]:
N = 50 


raw = rng.standard_normal((N, N))
fc_matrix = (raw @ raw.T) / N
np.fill_diagonal(fc_matrix, 1.0)

print(f"FC matrix shape:       {fc_matrix.shape}")
print(f"FC is symmetric:       {np.allclose(fc_matrix, fc_matrix.T)}")
print(f"FC condition number:   {np.linalg.cond(fc_matrix):.2f}")

## Step 2: Estimate Directed A Matrix (Spectral Inversion)

The Spectral Inversion solver addresses Dr. Agarwal's stability requirement:
- **Tikhonov damping** prevents division-by-zero on near-zero eigenvalues
- **Asymmetry injection** breaks the symmetry of the FC matrix to produce directed connectivity
- **Auto-stabilization** enforces spectral radius < 1 for downstream Gramian computation

In [ ]:
A_spectral, info_spectral = spectral_inversion_solver(fc_matrix, alpha=0.1, system='discrete')

is_stable, sr = check_schur_stability(A_spectral)

print("=== Spectral Inversion Solver ===")
print(f"  Spectral radius:     {info_spectral['spectral_radius']:.6f}  (must be < 1.0)")
print(f"  System stable:       {info_spectral['is_stable']}")
print(f"  Condition number:    {info_spectral['condition_number']:.2f}")
print(f"  A is asymmetric:     {not np.allclose(A_spectral, A_spectral.T)}")

## Step 3: Compare with Regularized MVAR Solver

In [ ]:

T = 600
timeseries = rng.standard_normal((N, T))

A_mvar, info_mvar = mvar_solver(timeseries, order=1, regularization='ridge', system='discrete')

print("=== Regularized MVAR Solver (Ridge) ===")
print(f"  Spectral radius:         {info_mvar['spectral_radius']:.6f}  (must be < 1.0)")
print(f"  System stable:           {info_mvar['is_stable']}")
print(f"  Post-hoc stabilization:  {info_mvar['stabilization_applied']}")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, A, title in zip(axes, [A_spectral, A_mvar], ["Spectral Inversion", "Regularized MVAR (Ridge)"]):
    im = ax.imshow(A, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
    ax.set_title(f"{title}\nSpectral Radius = {np.max(np.abs(np.linalg.eigvals(A))):.4f}",
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Source Node'); ax.set_ylabel('Target Node')
    plt.colorbar(im, ax=ax, label='Connection Weight')
plt.suptitle('NeuroSim Module B: Directed A Matrix Estimation', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('a_matrix_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Controllability Gramian

The Gramian quantifies the energy landscape of the state space.
Nodes with high Gramian eigenvalues are reachable with minimal energy.

In [ ]:

A_norm = normalize_matrix(A_spectral, system='discrete')
B = np.eye(N) 
T_horizon = 3

Wc = compute_gramian(A_norm, T=T_horizon, B=B, system='discrete')

print(f"Gramian shape:          {Wc.shape}")
print(f"Gramian is symmetric:   {np.allclose(Wc, Wc.T, atol=1e-8)}")
min_eig = np.linalg.eigvalsh(Wc).min()
print(f"Gramian min eigenvalue: {min_eig:.6f}  (must be >= 0 for PSD)")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(Wc, cmap='viridis', aspect='auto')
ax.set_title(f'Controllability Gramian (T={T_horizon})', fontsize=13, fontweight='bold')
ax.set_xlabel('Node'); ax.set_ylabel('Node')
plt.colorbar(im, ax=ax, label='Gramian Value')
plt.tight_layout()
plt.savefig('gramian.png', dpi=150)
plt.show()

## Step 5: Minimum Control Energy — State Transitions

In [ ]:

x_healthy   = np.zeros(N); x_healthy[:N//3]   = 1.0
x_aud       = np.zeros(N); x_aud[N//3:2*N//3] = 1.0
x_epilepsy  = np.zeros(N); x_epilepsy[2*N//3:]= 1.0

states = {'Healthy': x_healthy, 'AUD': x_aud, 'Epilepsy': x_epilepsy}
state_names = list(states.keys())
n_states = len(state_names)


energy_matrix = np.zeros((n_states, n_states))
for i, name_i in enumerate(state_names):
    for j, name_j in enumerate(state_names):
        E = minimum_energy(A_norm, T=T_horizon, B=B,
                           x0=states[name_i], xf=states[name_j],
                           system='discrete')
        energy_matrix[i, j] = E.sum()


fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(energy_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=state_names, yticklabels=state_names, ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_title('Pairwise Transition Energy (x0 → xf)', fontsize=13, fontweight='bold')
ax.set_xlabel('Target State (xf)')
ax.set_ylabel('Initial State (x0)')
plt.tight_layout()
plt.savefig('energy_landscape.png', dpi=150)
plt.show()

print("\nKey finding: Higher energy means harder to reach that state.")
print(f"Healthy → AUD:      {energy_matrix[0,1]:.4f} (addiction circuit reinforcement)")
print(f"Healthy → Epilepsy: {energy_matrix[0,2]:.4f} (seizure transition cost)")
print(f"AUD → Healthy:      {energy_matrix[1,0]:.4f} (restorative control effort)")